In [4]:

import os, json, datetime
import numpy as np
import pandas as pd
import fastf1 as ff1

from pathlib import Path
import datetime


In [6]:
# check the paths work from this notebook?


#i just want to check paths, trauma from when the cache went all over the place, wrong folder, went to tmp fml

#DONT NEED THE CACHE DIR BROTHER
# CACHE_DIR = Path(CACHE_DIR)   # <-- this is the missing line

# test_file = CACHE_DIR / "_cache_path_test.txt"
# with open(test_file, "w") as f:
#     f.write(f"Cache test written at {datetime.datetime.now().isoformat()}\n")

# print("Test file written to:", test_file)



OUT_DIR   = "../../data_processed"

OUT_DIR = Path(OUT_DIR)   # <-- this is the missing line
testp_file = OUT_DIR / "_parquets_path_test.txt"
with open(testp_file, "w") as f:
    f.write(f"Parquets test written at {datetime.datetime.now().isoformat()}\n")
    f.write(f"Works from the ml notebook fool\n")


print("Test file written to:", testp_file)




Test file written to: ..\..\data_processed\_parquets_path_test.txt


In [8]:
from pathlib import Path
import pandas as pd


DATASET_PATH = OUT_DIR / "preq_dataset.parquet"

df = pd.read_parquet(DATASET_PATH)
print(df.shape)

(2976, 16)


In [9]:
df

,year,round,event_name,country,location,driver,n_fp_laps,n_fp_sessions_participated,best_fp_lap_time_overall,avg_fp_lap_time,median_fp_lap_time,best_last_fp_lap_time,best_last_fp_s1,best_last_fp_s2,best_last_fp_s3,quali_position
0,2018,1,Australian Grand Prix,Australia,Melbourne,ALO,43,3,85.200,98.581512,96.2460,94.298,31.144,25.268,37.886,11.0
1,2018,1,Australian Grand Prix,Australia,Melbourne,BOT,63,3,84.159,98.692540,94.1740,94.174,31.326,25.150,37.698,10.0
2,2018,1,Australian Grand Prix,Australia,Melbourne,ERI,61,3,86.814,100.017279,90.7760,88.890,29.842,23.687,35.361,17.0
3,2018,1,Australian Grand Prix,Australia,Melbourne,GAS,69,3,85.945,99.882739,91.8910,94.990,31.187,25.521,38.282,20.0
4,2018,1,Australian Grand Prix,Australia,Melbourne,GRO,54,3,84.648,96.055611,90.5690,96.171,31.767,25.553,38.851,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2971,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,SAI,36,2,83.871,100.196750,90.0295,83.871,17.199,36.013,30.507,3.0
2972,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,STR,38,2,84.531,106.555658,90.9135,84.531,17.324,36.170,30.886,13.0
2973,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,TSU,38,2,84.343,111.313000,103.8435,84.343,17.303,36.247,30.793,11.0
2974,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,VER,35,2,83.844,105.660657,95.0140,83.844,17.206,36.156,30.482,5.0


In [10]:
df = df[df["quali_position"].notna()].reset_index(drop=True)

In [11]:
df

,year,round,event_name,country,location,driver,n_fp_laps,n_fp_sessions_participated,best_fp_lap_time_overall,avg_fp_lap_time,median_fp_lap_time,best_last_fp_lap_time,best_last_fp_s1,best_last_fp_s2,best_last_fp_s3,quali_position
0,2018,1,Australian Grand Prix,Australia,Melbourne,ALO,43,3,85.200,98.581512,96.2460,94.298,31.144,25.268,37.886,11.0
1,2018,1,Australian Grand Prix,Australia,Melbourne,BOT,63,3,84.159,98.692540,94.1740,94.174,31.326,25.150,37.698,10.0
2,2018,1,Australian Grand Prix,Australia,Melbourne,ERI,61,3,86.814,100.017279,90.7760,88.890,29.842,23.687,35.361,17.0
3,2018,1,Australian Grand Prix,Australia,Melbourne,GAS,69,3,85.945,99.882739,91.8910,94.990,31.187,25.521,38.282,20.0
4,2018,1,Australian Grand Prix,Australia,Melbourne,GRO,54,3,84.648,96.055611,90.5690,96.171,31.767,25.553,38.851,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2968,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,SAI,36,2,83.871,100.196750,90.0295,83.871,17.199,36.013,30.507,3.0
2969,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,STR,38,2,84.531,106.555658,90.9135,84.531,17.324,36.170,30.886,13.0
2970,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,TSU,38,2,84.343,111.313000,103.8435,84.343,17.303,36.247,30.793,11.0
2971,2024,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,VER,35,2,83.844,105.660657,95.0140,83.844,17.206,36.156,30.482,5.0


In [12]:
TARGET = "quali_position"
DROP_COLS = ["year","round","event_name","country","location","driver"]

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

In [13]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

groups = df["year"]   # LOYO by year

gkf = GroupKFold(n_splits=len(groups.unique()))

maes = []

for train_idx, test_idx in gkf.split(X, y, groups):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    )

    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    maes.append(mean_absolute_error(yte, preds))

print("Baseline LOYO MAE:", sum(maes)/len(maes))

Baseline LOYO MAE: 4.795286672026031


In [14]:
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective": "reg:squarederror",
        "random_state": 42,
    }

    maes = []

    for tr, te in gkf.split(X, y, groups):
        model = XGBRegressor(**params)
        model.fit(X.iloc[tr], y.iloc[tr])
        preds = model.predict(X.iloc[te])
        maes.append(mean_absolute_error(y.iloc[te], preds))

    return sum(maes) / len(maes)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)

c:\Users\aadha\anaconda3\envs\f1ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-01 11:31:31,666] A new study created in memory with name: no-name-a270fd9e-50b5-4d1f-9013-9993c8f0629a
[I 2026-03-01 11:31:33,461] Trial 0 finished with value: 5.035596213386298 and parameters: {'n_estimators': 727, 'max_depth': 3, 'learning_rate': 0.14016148827501465, 'subsample': 0.6113694427111653, 'colsample_bytree': 0.6640808580252927}. Best is trial 0 with value: 5.035596213386298.
[I 2026-03-01 11:31:34,283] Trial 1 finished with value: 4.893874385086852 and parameters: {'n_estimators': 468, 'max_depth': 2, 'learning_rate': 0.09036724498528163, 'subsample': 0.7379936185033411, 'colsample_bytree': 0.6186257020450407}. Best is trial 1 with value: 4.893874385086852.
[I 2026-03-01 11:31:36,427] Trial 2 finished with va

Best MAE: 4.705888009133067
Best params: {'n_estimators': 417, 'max_depth': 6, 'learning_rate': 0.015578346862257491, 'subsample': 0.8836224017995821, 'colsample_bytree': 0.9212351764810511}


In [15]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data_processed").exists():
            return p
    raise FileNotFoundError("Could not find repo root (folder containing data_processed).")

REPO_ROOT = find_repo_root(Path.cwd())
DATASET_PATH = REPO_ROOT / "data_processed" / "preq_dataset.parquet"

df = pd.read_parquet(DATASET_PATH)
df = df[df["quali_position"].notna()].reset_index(drop=True)

print("Loaded:", df.shape)
print("Years:", sorted(df["year"].unique()))

Loaded: (2973, 16)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [16]:
TARGET = "quali_position"
KEY_COLS = ["year","round","event_name","country","location","driver"]

X = df.drop(columns=KEY_COLS + [TARGET])
y = df[TARGET].astype(float)

In [17]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

def eval_loyo_mae(X, y, groups, params):
    gkf = GroupKFold(n_splits=len(np.unique(groups)))
    fold_mae = {}

    for tr, te in gkf.split(X, y, groups):
        yr = int(pd.Series(groups).iloc[te].unique()[0])

        model = XGBRegressor(**params)
        model.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[te])

        mae = mean_absolute_error(y.iloc[te], pred)
        fold_mae[yr] = mae

    return fold_mae

In [18]:
best_params = {
    "n_estimators": 417,
    "max_depth": 6,
    "learning_rate": 0.015578346862257491,
    "subsample": 0.8836224017995821,
    "colsample_bytree": 0.9212351764810511,
    "objective": "reg:squarederror",
    "random_state": 42,
}

groups = df["year"].values

fold_mae = eval_loyo_mae(X, y, groups, best_params)
print("LOYO MAE by year:")
for yr in sorted(fold_mae):
    print(yr, "->", fold_mae[yr])

print("Mean LOYO MAE:", sum(fold_mae.values()) / len(fold_mae))

LOYO MAE by year:
2018 -> 4.1547307843253725
2019 -> 4.103654438799078
2020 -> 4.919810310577573
2021 -> 4.813022782427846
2022 -> 5.158684067292647
2023 -> 5.013870523400502
2024 -> 4.777443157108443
Mean LOYO MAE: 4.705888009133067


In [19]:
final_model = XGBRegressor(**best_params)
final_model.fit(X, y)

# feature importance (gain-based)
importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(15))

best_last_fp_lap_time         0.139756
best_last_fp_s3               0.133125
best_last_fp_s2               0.130845
best_last_fp_s1               0.129456
best_fp_lap_time_overall      0.128264
n_fp_sessions_participated    0.102984
median_fp_lap_time            0.093400
avg_fp_lap_time               0.081460
n_fp_laps                     0.060710
dtype: float32
